# Phase 1 — Calibration

Goal: find the **anchor budget** `N` (env steps) that drives every subsequent run in EXPERIMENT_PLAN.md.

Procedure (from the plan):
1. Run a from-scratch **slippery** pilot with a generous budget. Identify the convergence step `N_slip`.
2. Run a from-scratch **normal** pilot with the same budget. Identify `N_norm`.
3. Set `N = N_norm`. If `2N < N_slip`, bump `N` so `2N` covers slippery convergence and document the bump.
4. Pick the eval cadence `X` so eval fires ~30–50 times across a budget-`N` run.

Outputs: writes `runs/_phase1_calibration.json` with the chosen `N` and `X` for Phase 2 to consume.

## 0. Setup

In [ ]:
import json, math, os, pathlib
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

from experiment.config import ExperimentConfig, FRICTION_NORMAL, FRICTION_SLIPPERY
from experiment.train import train
from experiment.run_io import read_jsonl
from experiment.evaluate import aggregate_by_friction

# ------------------------------------------------------------------
# Knobs for the pilot. Bump max_env_steps if either curve hasn't
# plateaued by the end -- the whole point is to *see* the plateau, so
# erring on the long side is fine. Eval cadence is also relatively
# tight here so the convergence-detection heuristic has enough points
# to be reliable.
# ------------------------------------------------------------------
PILOT_MAX_ENV_STEPS  = 300_000      # ~3x what we expect to need
PILOT_EVAL_EVERY     = 5_000       # ~60 eval checkpoints per pilot
PILOT_EVAL_N_EP      = 1           # episodes per (map, friction) cell at eval time
PILOT_SEED           = 0
PILOT_SAVE_DIR       = "runs"
PILOT_DEVICE         = "cpu"        # set to "cuda" if a GPU is available

SLIPPERY_RUN_NAME = "phase1_pilot_slippery"
NORMAL_RUN_NAME   = "phase1_pilot_normal"

CALIBRATION_OUT = pathlib.Path(PILOT_SAVE_DIR) / "_phase1_calibration.json"

def _run_dir(name, seed): return pathlib.Path(PILOT_SAVE_DIR) / name / f"seed_{seed}"
def _eval_log(name, seed): return _run_dir(name, seed) / "eval_log.jsonl"

def _exists(name, seed): return _eval_log(name, seed).exists()

print(f"slippery pilot exists: {_exists(SLIPPERY_RUN_NAME, PILOT_SEED)}  ({_run_dir(SLIPPERY_RUN_NAME, PILOT_SEED)})")
print(f"normal   pilot exists: {_exists(NORMAL_RUN_NAME,   PILOT_SEED)}  ({_run_dir(NORMAL_RUN_NAME,   PILOT_SEED)})")

## 1. Pilot — from-scratch slippery

Single seed, generous budget. This pins `N_slip` (the harder task's convergence point).

**Wall-clock warning**: 800k env steps on CPU is on the order of 30–90 minutes (depending on how often episodes wall-hit early). Re-running the cell will skip training if a checkpoint exists — delete the run directory to force a redo.

In [ ]:
cfg_slip = ExperimentConfig(
    run_name             = SLIPPERY_RUN_NAME,
    seed                 = PILOT_SEED,
    save_dir             = PILOT_SAVE_DIR,
    device               = PILOT_DEVICE,
    friction_mode        = "fixed",
    friction             = FRICTION_SLIPPERY,
    max_env_steps        = PILOT_MAX_ENV_STEPS,
    eval_every_env_steps = PILOT_EVAL_EVERY,
    eval_n_episodes      = PILOT_EVAL_N_EP,
    eval_frictions       = (FRICTION_NORMAL, FRICTION_SLIPPERY),
)

if _exists(SLIPPERY_RUN_NAME, PILOT_SEED):
    print(f"[skip] {SLIPPERY_RUN_NAME}/seed_{PILOT_SEED} already exists. Delete it to retrain.")
else:
    train(cfg_slip)

## 2. Pilot — from-scratch normal

Same budget on normal friction. This pins `N_norm`.

In [ ]:
cfg_norm = ExperimentConfig(
    run_name             = NORMAL_RUN_NAME,
    seed                 = PILOT_SEED,
    save_dir             = PILOT_SAVE_DIR,
    device               = PILOT_DEVICE,
    friction_mode        = "fixed",
    friction             = FRICTION_NORMAL,
    max_env_steps        = PILOT_MAX_ENV_STEPS,
    eval_every_env_steps = PILOT_EVAL_EVERY,
    eval_n_episodes      = PILOT_EVAL_N_EP,
    eval_frictions       = (FRICTION_NORMAL, FRICTION_SLIPPERY),
)

if _exists(NORMAL_RUN_NAME, PILOT_SEED):
    print(f"[skip] {NORMAL_RUN_NAME}/seed_{PILOT_SEED} already exists. Delete it to retrain.")
else:
    train(cfg_norm)

## 3. Load + plot the pilot eval curves

For each pilot we pull out the **self-task** eval cells (slippery cells for the slippery pilot, normal cells for the normal pilot), average across maps per checkpoint, and plot.

In [ ]:
def load_self_task_curve(run_name, seed, target_friction):
    """Return (env_steps, mean_return_across_maps) for the target friction."""
    log = read_jsonl(_eval_log(run_name, seed))
    steps, vals = [], []
    for rec in log:
        # cells is a list of dicts: {map, friction, return_mean, ...}
        same_f = [c for c in rec["cells"]
                  if math.isclose(c["friction"], target_friction)]
        if not same_f:
            continue
        steps.append(rec["env_steps"])
        vals.append(float(np.mean([c["return_mean"] for c in same_f])))
    return np.array(steps), np.array(vals)

slip_steps, slip_vals = load_self_task_curve(SLIPPERY_RUN_NAME, PILOT_SEED, FRICTION_SLIPPERY)
norm_steps, norm_vals = load_self_task_curve(NORMAL_RUN_NAME,   PILOT_SEED, FRICTION_NORMAL)

def smooth(y, window=5):
    if len(y) < window:
        return y.copy()
    pad = window // 2
    kernel = np.ones(window) / window
    out = np.convolve(y, kernel, mode="same")
    # Edge effects: snap the first/last few back to the raw values so we
    # don't get a misleading dip at the right edge of the plot.
    out[:pad] = y[:pad]
    out[-pad:] = y[-pad:]
    return out

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (steps, vals, label, color) in zip(axes, [
    (slip_steps, slip_vals, "slippery pilot (eval at f=0.95)", "tab:red"),
    (norm_steps, norm_vals, "normal pilot (eval at f=0.10)",   "tab:blue"),
]):
    if len(steps) == 0:
        ax.set_title(f"{label}: no eval log found")
        continue
    ax.plot(steps, vals, alpha=0.35, color=color, label="eval mean (per ckpt)")
    ax.plot(steps, smooth(vals), color=color, lw=2, label="smoothed (w=5)")
    ax.set_xlabel("env steps")
    ax.set_ylabel("mean greedy return")
    ax.set_title(label)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Identify convergence steps (`N_slip`, `N_norm`)

Heuristic, applied to each smoothed curve:

1. **Asymptote** = mean of the last 20% of the smoothed eval values.
2. **Convergence threshold** = 90% of (asymptote − starting value), interpreted as a level the curve must reach.
3. `N_converge` = first env-step where the smoothed value reached the threshold *and* didn't drop back below it for the rest of the run.

Print + plot the picks. Override the values in the next cell if the heuristic looks wrong.

In [ ]:
def find_convergence_step(steps, vals, asymptote_frac=0.20, threshold_frac=0.90):
    """Heuristic: first step where smoothed value reaches `threshold_frac`
    of the (asymptote - start) gain, and stays at-or-above it afterwards.
    Returns (n_converge_step, asymptote, threshold)."""
    if len(steps) < 5:
        return None, float("nan"), float("nan")
    s = smooth(vals)
    tail = s[max(1, int(len(s) * (1 - asymptote_frac))):]
    asymptote = float(np.mean(tail))
    start = float(s[0])
    threshold = start + threshold_frac * (asymptote - start)
    above = s >= threshold
    # Walk from the right edge backwards to find the longest tail that's
    # all-True, then the convergence step is the first index of that tail.
    i = len(s) - 1
    while i > 0 and above[i]:
        i -= 1
    converge_idx = i + 1  # first index of the contiguous "above" tail
    if converge_idx >= len(s):
        return None, asymptote, threshold
    return int(steps[converge_idx]), asymptote, threshold

n_slip, asymp_slip, thr_slip = find_convergence_step(slip_steps, slip_vals)
n_norm, asymp_norm, thr_norm = find_convergence_step(norm_steps, norm_vals)

print(f"slippery: N_slip ≈ {n_slip:,}   asymptote={asymp_slip:.1f}   threshold={thr_slip:.1f}" if n_slip else "slippery: NOT CONVERGED -- bump PILOT_MAX_ENV_STEPS and re-run")
print(f"normal:   N_norm ≈ {n_norm:,}   asymptote={asymp_norm:.1f}   threshold={thr_norm:.1f}" if n_norm else "normal:   NOT CONVERGED -- bump PILOT_MAX_ENV_STEPS and re-run")

# Visual confirmation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (steps, vals, n_conv, asymp, thr, label, color) in zip(axes, [
    (slip_steps, slip_vals, n_slip, asymp_slip, thr_slip, "slippery", "tab:red"),
    (norm_steps, norm_vals, n_norm, asymp_norm, thr_norm, "normal",   "tab:blue"),
]):
    if len(steps) == 0: continue
    ax.plot(steps, vals, alpha=0.35, color=color, label="eval mean")
    ax.plot(steps, smooth(vals), color=color, lw=2, label="smoothed")
    if not math.isnan(asymp):
        ax.axhline(asymp, color="k", ls="--", alpha=0.5, label=f"asymptote ≈ {asymp:.1f}")
        ax.axhline(thr,   color="gray", ls=":",  alpha=0.5, label=f"90% threshold ≈ {thr:.1f}")
    if n_conv is not None:
        ax.axvline(n_conv, color="green", lw=2, alpha=0.7, label=f"N ≈ {n_conv:,}")
    ax.set_xlabel("env steps"); ax.set_ylabel("mean greedy return")
    ax.set_title(f"{label} pilot: convergence detection")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4b. Manual override (optional)

If the heuristic's pick disagrees with what you see in the plot, set the values explicitly here. **Skip this cell** to keep the heuristic picks.

In [ ]:
# Uncomment + set to override
n_slip = 500_000
n_norm = 150_000

print(f"using N_slip = {n_slip:,}")
print(f"using N_norm = {n_norm:,}")

## 5. Lock in `N`

Rule: `N = N_norm`, but if `2N < N_slip` then bump `N` so `2N` covers slippery convergence (we want the baseline run at budget `2N` to actually plateau). The bump is documented in the methodology section of the report.

In [ ]:
if n_slip is None or n_norm is None:
    raise RuntimeError("Both pilots must converge before locking N. "
                       "Bump PILOT_MAX_ENV_STEPS and re-run the pilots.")

N_candidate = n_norm
if 2 * N_candidate < n_slip:
    N = int(math.ceil(n_slip / 2))
    bump_note = (f"BUMPED: 2 * N_norm ({2 * n_norm:,}) < N_slip ({n_slip:,}); "
                 f"set N = ceil(N_slip / 2) = {N:,} so 2N covers slippery convergence.")
else:
    N = N_candidate
    bump_note = (f"OK: 2 * N_norm ({2 * n_norm:,}) >= N_slip ({n_slip:,}). "
                 f"No bump needed.")

# Round N up to the nearest 10k for tidy run names and plot axes.
def _round_to_nice(x):
    return int(math.ceil(x / 10_000) * 10_000)
N = _round_to_nice(N)

print(f"N_norm = {n_norm:,}")
print(f"N_slip = {n_slip:,}")
print(f"--> {bump_note}")
print(f"--> N = {N:,}  (used as pretrain / fine-tune budget; 2N = {2*N:,} for baselines)")

## 6. Lock in `X` (eval cadence)

Target 30–50 eval points across a budget-`N` run. We also estimate the per-eval cost from the slippery pilot's wall_seconds field so the cadence can't accidentally make eval dominate wall-clock.

In [ ]:
TARGET_EVAL_POINTS = 40
X_candidate = _round_to_nice(N / TARGET_EVAL_POINTS)

# Sanity-check eval cost: read wall_seconds from successive eval records
# in the slippery pilot and infer the *per-eval* cost (delta wall_seconds
# minus the training wall_seconds between eval points).
slip_log = read_jsonl(_eval_log(SLIPPERY_RUN_NAME, PILOT_SEED))
if len(slip_log) >= 2:
    deltas_wall  = np.array([slip_log[i]["wall_seconds"] - slip_log[i-1]["wall_seconds"]
                              for i in range(1, len(slip_log))])
    deltas_steps = np.array([slip_log[i]["env_steps"]    - slip_log[i-1]["env_steps"]
                              for i in range(1, len(slip_log))])
    # Rough breakdown: most of `delta_wall` is training time, plus one
    # eval pass. Solve for eval cost via the smallest delta (likely the
    # one where training was fastest, so eval dominates the residual).
    # This is approximate -- a real estimate would time the eval call.
    sps_estimate = float(np.percentile(deltas_steps / deltas_wall, 75))
    avg_train_per_chunk = PILOT_EVAL_EVERY / sps_estimate
    eval_cost_estimate = float(np.maximum(deltas_wall - avg_train_per_chunk, 0).mean())
    full_run_evals = N / X_candidate
    eval_overhead = eval_cost_estimate * full_run_evals
    train_wall    = N / sps_estimate
    print(f"steps/sec estimate (75th pct): {sps_estimate:>6.0f}")
    print(f"per-eval cost estimate:        {eval_cost_estimate:>6.2f} s")
    print(f"full {N:,}-step run estimate:  train {train_wall/60:.1f} min, eval {eval_overhead/60:.1f} min ({100*eval_overhead/(train_wall+eval_overhead):.1f}% overhead)")

X = X_candidate
print(f"\n--> X = {X:,} env steps (~{N // X} eval points per run)")

## 7. Write calibration result

Saved to `runs/_phase1_calibration.json`. Phase 2's HP sweep code reads this file so the chosen `N`/`X` are the single source of truth.

In [ ]:
calibration = {
    "N":      int(N),
    "X":      int(X),
    "N_norm": int(n_norm),
    "N_slip": int(n_slip),
    "asymptote_normal":   float(asymp_norm),
    "asymptote_slippery": float(asymp_slip),
    "pilot": {
        "seed":           PILOT_SEED,
        "max_env_steps":  PILOT_MAX_ENV_STEPS,
        "eval_every":     PILOT_EVAL_EVERY,
        "eval_n_ep":      PILOT_EVAL_N_EP,
        "slippery_run":   SLIPPERY_RUN_NAME,
        "normal_run":     NORMAL_RUN_NAME,
    },
    "notes": bump_note,
}

CALIBRATION_OUT.parent.mkdir(parents=True, exist_ok=True)
CALIBRATION_OUT.write_text(json.dumps(calibration, indent=2))
print(f"wrote {CALIBRATION_OUT}")
print(json.dumps(calibration, indent=2))

## 8. Catastrophic-forgetting sanity check (free bonus)

Both pilots evaluate on **both** friction conditions at every checkpoint, so we already have the cross-evaluation. Plot the off-task curves to confirm: trains-on-normal should be terrible at slippery (cross-friction generalization gap), and the same other way around. If this isn't visible, something's wrong with the env or the eval pipeline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (run_name, on_friction, off_friction, label) in zip(axes, [
    (SLIPPERY_RUN_NAME, FRICTION_SLIPPERY, FRICTION_NORMAL, "slippery pilot"),
    (NORMAL_RUN_NAME,   FRICTION_NORMAL,   FRICTION_SLIPPERY, "normal pilot"),
]):
    on_steps,  on_vals  = load_self_task_curve(run_name, PILOT_SEED, on_friction)
    off_steps, off_vals = load_self_task_curve(run_name, PILOT_SEED, off_friction)
    if len(on_steps):
        ax.plot(on_steps,  smooth(on_vals),  label=f"on-task (f={on_friction})",  lw=2)
    if len(off_steps):
        ax.plot(off_steps, smooth(off_vals), label=f"off-task (f={off_friction})", lw=2, ls="--")
    ax.set_xlabel("env steps"); ax.set_ylabel("mean greedy return")
    ax.set_title(f"{label}: on-task vs off-task eval")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Phase 1 summary

If you got here without errors and the plots make sense:

- `N` is locked in `runs/_phase1_calibration.json`.
- `X` is set so eval doesn't dominate wall-clock.
- The pilots show a real generalization gap between friction regimes (sanity-check passed).

**Pre-flight items still to do before Phase 2** (per EXPERIMENT_PLAN §0/pre-flight checklist, none are blocking but all are cheap):

1. Confirm the curriculum schedule actually mutates `env.friction` per episode on a small dry-run (already covered by `test_phase0.test_curriculum_applied_via_MultiMapEnv`).
2. Time one full eval pass (3 maps × 2 frictions × 10 eps) end-to-end so the per-run wall-clock estimate above is grounded in reality.
3. Estimate total wall-clock for Phase 2: 7 conditions × 6 HP configs × 3 seeds × N env-steps. Decide if you need to cut search budget (`max_env_steps` at 50% of N is usually fine for AUC).

Now you're ready to launch Phase 2.